# 3 4 count analysis deleted posts

Peer-review copy. Inputs are pseudonymized processed data; see README.md and DATA_DICTIONARY.md. Original preparation/collection steps are outside this package.

In [ ]:
from review_setup import prepare_workspace
prepare_workspace()
from review_setup import initialize_stata
initialize_stata()


In [ ]:
import pandas as pd
# Processed deleted-post count panels, with event and account keys.
ts_posts_deleted = pd.read_parquet('data/deleted_posts_daily.parquet')
ts_posts_deleted_weekly = pd.read_parquet('data/deleted_posts_weekly.parquet')
print('Deleted-post events with valid counts:', ts_posts_deleted['tweet_id'].nunique())


#### Daily DiD estimation

In [ ]:
from utils import get_source_corrections, StataDiD

source_corrections = get_source_corrections(window_days=4).query('treated==1').reset_index(drop=True)
main_ts_posts = pd.read_parquet('data/ts_posts_daily.parquet.gzip').query('-4<=days_from_display<=4').reset_index(drop=True)
main_ts_posts = main_ts_posts.merge(source_corrections, on='tweet_id', how='inner')
print(main_ts_posts.shape)
main_ts_posts.head()

In [ ]:
ts_posts_deleted['treated'] = 1
main_ts_posts['treated'] = 0
ts_posts_daily = pd.concat([main_ts_posts[['tweet_id', 'days_from_display', 'user_id', 'tweet_count', 'treated']], ts_posts_deleted.query('-4<=days_from_display<=4')[['tweet_id', 'days_from_display', 'user_id', 'tweet_count', 'treated']]], ignore_index=True, axis=0)
ts_posts_daily['account_cluster'] = ts_posts_daily['user_id'].astype(str)
ts_posts_daily.head()

In [ ]:
stata_did = StataDiD(ts_posts_daily, 'treated', 'tweet_id', 'days_from_display', -4, 4)

In [ ]:
stata_did.multi_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster')

In [ ]:
from utils import get_coefs
means_mp_atts, lls_mp_atts, hls_mp_atts = get_coefs(reg='ppmlhdfe', start_line=8, end_line=15, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
stata_did.tw_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster')

In [ ]:
mean_agg_att, ll_agg_att, hl_agg_att = get_coefs(reg='ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
from utils import att_plot
att_plot(-4, 4, 4, means_mp_atts, lls_mp_atts, hls_mp_atts, mean_agg_att, ll_agg_att, hl_agg_att, 'Daily', 'Days from note display', 'Estimated DiD effect\n(deleted vs. non-deleted)', 'did_posts_daily_deleted', ylims=(-0.862, 1.862), legend_ncol=1, legend_title='All corrections')

#### Weekly DiD estimation

In [ ]:
source_corrections = get_source_corrections(window_days=28).query('treated==1').reset_index(drop=True)
main_ts_posts_weekly = pd.read_parquet('data/ts_posts_weekly.parquet.gzip')
main_ts_posts_weekly = main_ts_posts_weekly.merge(source_corrections, on='tweet_id', how='inner')
print(main_ts_posts_weekly.shape)
main_ts_posts_weekly.head()

In [ ]:
ts_posts_deleted_weekly['treated'] = 1
main_ts_posts_weekly['treated'] = 0
ts_posts_weekly = pd.concat([main_ts_posts_weekly[['tweet_id', 'weeks_from_display', 'user_id', 'tweet_count', 'treated']], ts_posts_deleted_weekly[['tweet_id', 'weeks_from_display', 'user_id', 'tweet_count', 'treated']]], ignore_index=True, axis=0)
ts_posts_weekly['account_cluster'] = ts_posts_weekly['user_id'].astype(str)
ts_posts_weekly.head()

In [ ]:
stata_did = StataDiD(ts_posts_weekly, 'treated', 'tweet_id', 'weeks_from_display', -4, 4)

In [ ]:
stata_did.multi_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster')

In [ ]:
means_mp_atts, lls_mp_atts, hls_mp_atts = get_coefs(reg='ppmlhdfe', start_line=8, end_line=15, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
stata_did.tw_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster')

In [ ]:
mean_agg_att, ll_agg_att, hl_agg_att = get_coefs(reg='ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
att_plot(-4, 4, 4, means_mp_atts, lls_mp_atts, hls_mp_atts, mean_agg_att, ll_agg_att, hl_agg_att, 'Weekly', 'Weeks from note display', 'Estimated DiD effect\n(deleted vs. non-deleted)', 'did_posts_weekly_deleted', ylims=(-0.862, 1.862), legend_ncol=1, legend_title='All corrections')